# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [2]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch
import random

import warnings
warnings.filterwarnings("ignore")

In [3]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [4]:
# CONFIGURATION
DATA_ROOT = '../../data/raw/messy_mashup/genres_stems' # Change this to your data directory
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'] # Make the list of all genres available
STEMS = {
    'drums': 'drums.wav',
    'vocals': 'vocals.wav',
    'bass': 'bass.wav',
    'other': 'other.wav'
} # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [20]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

    # Counts for questions
    corrupted_count = 0
    less_than_lower_limit=0
    greater_than_upper_limit=0

    MB=1024*1024
    lower_limit=5.0491*MB
    upper_limit=5.0493*MB

    ## Genre -> Song -> 4*Stems

    # Iterate through Genres
    for genre in GENRES:

        genre_path = os.path.join(root_dir, genre)
        # Check: if genre folder exists
        if not os.path.exists(genre_path):
            raise ValueError(f"Genre folder does not exist: {genre_path}")
        
        #Get list of songs in the genre
        songs=os.listdir(genre_path)

        valid_songs = []

        for song in songs:
            
            song_path = os.path.join(genre_path, song)
            # Checking: if song folder exists
            if not os.path.isdir(song_path):
                print(f"Warning: Song folder does not exist: {song_path}. Skipping.")
                continue

            stem_flies=[] # List to hold paths of stem files for the current song
            is_corrupt=False

            for stem in STEMS:
                stem_path = os.path.join(song_path, STEMS[stem])
                
                # Check: if stem file exists
                if not os.path.isfile(stem_path):
                    print(f"Warning: Stem file does not exist: {stem_path}. Skipping song: {song_path}.")
                    is_corrupt=True
                    break

                size=os.path.getsize(stem_path)

                # CHECK : Corruption (Is any file too small? (less than 4kb))
                if size < 4*1024: # 4KB
                    print(f"Warning: Stem file is too small (potentially corrupt): {stem_path}. Skipping song: {song_path}.")
                    is_corrupt=True
                    break
                
                # size checks
                # Count for lower & upper limit
                if size < lower_limit:
                    less_than_lower_limit+=1
                elif size > upper_limit:
                    greater_than_upper_limit+=1
                
                stem_flies.append(stem_path)

            # CHECK : Completeness (Does it have all stems?)
            if not is_corrupt:
                valid_songs.append(song_path)
            else:
                corrupted_count+=1

        # Stratified Shuffle Split  
        rng.shuffle(valid_songs)

        split_idx = int(len(valid_songs) * (1 - val_split)) # index to split train and val
        train_songs = valid_songs[:split_idx]
        val_songs = valid_songs[split_idx:]
     #-------------------------------------------------------------------------

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            for song in song_list:
                for stem_key, stem_name in STEMS.items():
                    stem_path=os.path.join(song, stem_name)
                    # genre -> stem -> list of paths
                    target_dict[genre][stem_key].append(stem_path)
        
        # Adding the valid songs to respective dictionaries
        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)
        
    print(f"Total corrupted songs: {corrupted_count}")
    print(f"Total songs with size <5.0491MB: {less_than_lower_limit}")
    print(f"Total songs with size >5.0493MB: {greater_than_upper_limit}")

    print(f"Diff: {abs(greater_than_upper_limit - less_than_lower_limit)}")

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

Total corrupted songs: 0
Total songs with size <5.0491MB: 1256
Total songs with size >5.0493MB: 184
Diff: 1072


In [21]:
# Q: What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?
len(tr['reggae']['drums']) - len(val['country']['vocals'])

66

In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    # ------------------- write your code here -------------------------------
    total_files = 0

    for genre, stems in dataset_dict.items():
        for stem in stems:
            stem_name = stem.replace('.wav', '')
            for file_path in stems[stem]:
                # ---- COUNT TOTAL FILES ----
                total_files += 1
                
                # Load Audio with sr sample rate
                y, _ = librosa.load(file_path, sr=sr) # y-> audio time series data aplitude vs time

                # Getting total audio duration in seconds
                total_duration= librosa.get_duration(y=y, sr=sr)
                
                # Find Non-Silent Intervals
                intervals = librosa.effects.split(y, top_db=top_db)
                
                silence_type = [] # To store the type of silence (A, B, C, D)
                max_silence = 0.0
                total_silence = 0.0

                # CASE A: Fully silent
                if len(intervals) == 0:
                    max_silence = total_duration
                    total_silence = total_duration
                    silence_type.append("FULL")
                else:
                    # CASE B: START silence
                    if intervals[0][0] > 0:
                        silence_duration = intervals[0][0] / sr # Convert to seconds
                        max_silence = max(silence_duration, max_silence)
                        total_silence += silence_duration
                        silence_type.append("START")

                    # CASE C: END silence
                    if intervals[-1][1] < len(y):
                        silence_duration = (len(y) - intervals[-1][1]) / sr # Convert to seconds
                        max_silence = max(silence_duration, max_silence)
                        total_silence += silence_duration
                        silence_type.append("END")

                    # CASE D: MIDDLE silence
                    # Iterating through pairs of consecutive intervals to find middle silences
                    for i in range(len(intervals) - 1):
                        start = intervals[i][1] # End of current interval
                        end = intervals[i+1][0] # Start of next interval
                        if end > start:
                            silence_duration = (end - start) / sr # Convert to seconds
                            max_silence = max(silence_duration, max_silence)
                            total_silence += silence_duration
                            silence_type.append("MIDDLE")
        

                # Store result
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Total_Silence_Sec": round(total_silence, 2),
                        "Silence_Location": ", ".join(silence_type),
                        "File_Path": file_path
                    })
    #-------------------------------------------------------------------------
    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

In [ ]:
# --- RESULTS ANALYSIS ---

# ------------------- write your code here -------------------------------
print(f"Total number of sound files having silence greater than equal to 5 secs. {len(df_silence)}")

# Hint: Create a pivot Table: Count by Genre vs Stem
df_silence_pivot = df_silence.pivot_table(index="Genre", columns="Stem", aggfunc="size", fill_value=0)

print(f"Total number of sound tracks in Vocals where silence >= 5 secs: {df_silence_pivot['vocals'].sum()}")

print(f"What's the average Silence Length in Vocals (in secs). {df_silence[df_silence['Stem']=='vocals']['Max_Silence_Sec'].mean():.2f}")

print(f"Total number of drums sound tracks in jazz where silence >= 5 secs: {df_silence_pivot.loc['jazz', 'drums']}")

print(f"Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle. \
      {len(df_silence[(df_silence['Genre']=='jazz') &
                        (df_silence['Stem']=='drums') &
                        (df_silence['Silence_Location']=='MIDDLE')])}")

print(f"Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10. \
      {len(df_silence[(df_silence['Genre']=='jazz') &
                       (df_silence['Stem']=='drums') &
                         (df_silence['Max_Silence_Sec']>=10)])}")
#-------------------------------------------------------------------------

Total number of sound files having silence greater than equal to 5 secs. 678
Total number of sound tracks in Vocals where silence >= 5 secs: 305
What's the average Silence Length in Vocals (in secs). 12.48
Total number of drums sound tracks in jazz where silence >= 5 secs: 21
Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle.       0
Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10. 8


In [ ]:
stems_audio = []
try:
    for key in STEM_KEYS:
      pass
    # ------------------- write your code here -------------------------------
    # Load audio (Duration 5.0s for speed/consistency)
    #-------------------------------------------------------------------------

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

In [ ]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack =

# Mix the stems by summing them element-wise
mix_raw =

# Calculate RMS Amplitude MANUALLY
rms_val =

#Peak Normalization
max_val =

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------
